# Transformer Architecture — Exercises

8 exercises covering the mathematical foundations of the Transformer architecture.

| Format | Description |
|---|---|
| **Problem** | Markdown cell with task description |
| **Your Solution** | Code cell with scaffolding |
| **Solution** | Code cell with reference solution and checks |

### Difficulty Levels

| Level | Exercises | Focus |
|---|---|---|
| ★ | 1-3 | Core mechanics |
| ★★ | 4-6 | Deeper theory |
| ★★★ | 7-8 | AI / ML applications |

### Topic Map

| Topic | Exercise |
|---|---|
| Scaled dot-product attention | 1 |
| Parameter counting | 2 |
| Causal masking | 3 |
| RoPE | 4 |
| Variance scaling | 5 |
| Multi-head attention | 6 |
| FlashAttention (online softmax) | 7 |
| LoRA adaptation | 8 |

In [ ]:
import numpy as np
import numpy.linalg as la

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

np.set_printoptions(precision=6, suppress=True)
np.random.seed(42)

def header(title):
    print('\n' + '=' * len(title))
    print(title)
    print('=' * len(title))

def check_close(name, got, expected, tol=1e-8):
    ok = np.allclose(got, expected, atol=tol, rtol=tol)
    print(f"{'PASS' if ok else 'FAIL'} — {name}")
    if not ok:
        print('  expected:', expected)
        print('  got     :', got)
    return ok

def check_true(name, cond):
    print(f"{'PASS' if cond else 'FAIL'} — {name}")
    return cond

def softmax(x, axis=-1):
    x_max = np.max(x, axis=axis, keepdims=True)
    e_x = np.exp(x - x_max)
    return e_x / np.sum(e_x, axis=axis, keepdims=True)

print('Setup complete.')

---

## Exercise 1: Scaled Dot-Product Attention (★)

Implement the core attention function from scratch.

Given $Q, K, V \in \mathbb{R}^{n \times d}$, compute:
$$\operatorname{Attention}(Q, K, V) = \operatorname{softmax}\left(\frac{QK^\top}{\sqrt{d}}\right) V$$

(a) Compute the score matrix $S = QK^\top / \sqrt{d}$
(b) Apply row-wise softmax to get attention weights $A$
(c) Compute the output $O = AV$
(d) Verify row sums = 1 and all weights non-negative
(e) Test with $n = 4$, $d = 8$

In [ ]:
# Exercise 1: Your Solution

def attention(Q, K, V):
    """Scaled dot-product attention."""
    # YOUR CODE HERE
    d_k = Q.shape[-1]
    scores = None  # Step (a): compute QK^T / sqrt(d_k)
    weights = None  # Step (b): apply softmax row-wise
    output = None   # Step (c): compute weights @ V
    return output, weights

# Test data
n, d = 4, 8
Q = np.random.randn(n, d)
K = np.random.randn(n, d)
V = np.random.randn(n, d)

result = attention(Q, K, V)
print(result)

In [ ]:
# Exercise 1: Solution

def attention(Q, K, V):
    """Scaled dot-product attention."""
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)
    weights = softmax(scores, axis=-1)
    output = weights @ V
    return output, weights

n, d = 4, 8
np.random.seed(42)
Q = np.random.randn(n, d)
K = np.random.randn(n, d)
V = np.random.randn(n, d)

output, weights = attention(Q, K, V)

header('Exercise 1: Scaled Dot-Product Attention')
print(f'Score matrix shape: {(Q @ K.T).shape}')
print(f'Attention weights shape: {weights.shape}')
print(f'Output shape: {output.shape}')

check_close('row sums = 1', weights.sum(axis=1), np.ones(n))
check_true('all weights >= 0', np.all(weights >= 0))
check_true('output has correct shape', output.shape == (n, d))

print('\nTakeaway: Attention is softmax(QK^T/sqrt(d))V — a weighted average '
      'of values with data-dependent weights.')

---

## Exercise 2: Parameter Counting (★)

For a single Transformer block with $d_{\text{model}} = 512$, $h = 8$, $d_{ff} = 2048$:

(a) Count MHA parameters ($W_Q, W_K, W_V, W_O$)
(b) Count ReLU FFN parameters ($W_1, W_2$ + biases)
(c) Count SwiGLU FFN parameters ($W_1, W_2, W_3$, $d_{ff} = \lfloor 8/3 \cdot 512 \rfloor$)
(d) Count 2 RMSNorm layers
(e) Verify the $12d^2$ approximation

In [ ]:
# Exercise 2: Your Solution

d_model = 512
h = 8
d_ff_relu = 2048
d_ff_swiglu = int(8/3 * d_model)

mha_params = None        # (a) 4 * d_model * d_model
ffn_relu_params = None   # (b) 2 * d_model * d_ff + d_ff + d_model
ffn_swiglu_params = None # (c) 3 * d_model * d_ff_swiglu
norm_params = None       # (d) 2 * d_model

print(f'MHA params: {mha_params}')
print(f'FFN (ReLU) params: {ffn_relu_params}')
print(f'FFN (SwiGLU) params: {ffn_swiglu_params}')
print(f'Norm params: {norm_params}')

In [ ]:
# Exercise 2: Solution

d_model = 512
h = 8
d_ff_relu = 2048
d_ff_swiglu = int(8/3 * d_model)

mha_params = 4 * d_model * d_model
ffn_relu_params = 2 * d_model * d_ff_relu + d_ff_relu + d_model
ffn_swiglu_params = 3 * d_model * d_ff_swiglu
norm_params = 2 * d_model
total = mha_params + ffn_relu_params + norm_params
total_swiglu = mha_params + ffn_swiglu_params + norm_params

header('Exercise 2: Parameter Counting')
print(f'MHA params:        {mha_params:>12,} = 4 * {d_model}^2')
print(f'FFN (ReLU) params: {ffn_relu_params:>12,}')
print(f'FFN (SwiGLU) params:{ffn_swiglu_params:>11,} (d_ff={d_ff_swiglu})')
print(f'Norm params:       {norm_params:>12,}')
print(f'Total (ReLU):      {total:>12,}')
print(f'Total (SwiGLU):    {total_swiglu:>12,}')

approx = 12 * d_model**2
check_close('12d^2 approximation (ReLU)', total, approx, tol=approx*0.1)
check_true('MHA = 4d^2', mha_params == 4 * d_model**2)
check_true('FFN ReLU ~ 8d^2', abs(ffn_relu_params - 8*d_model**2) < 5*d_model)

print('\nTakeaway: FFN has 2x the parameters of attention. '
      'Total per block ~ 12d^2, independent of number of heads.')

---

## Exercise 3: Causal Masking (★)

(a) Create a causal mask for $n = 6$
(b) Implement masked attention
(c) Show position $i$'s output depends only on positions $\le i$
(d) Verify the attention matrix is lower-triangular

In [ ]:
# Exercise 3: Your Solution

def create_causal_mask(n):
    # YOUR CODE HERE: return mask with -1e9 for future positions
    pass

def masked_attention(Q, K, V, mask):
    # YOUR CODE HERE: add mask to scores before softmax
    pass

n = 6
mask = create_causal_mask(n)
print(mask)

In [ ]:
# Exercise 3: Solution

def create_causal_mask(n):
    mask = np.full((n, n), -1e9)
    mask = np.triu(mask, k=1)
    return mask

def masked_attention(Q, K, V, mask):
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k) + mask
    weights = softmax(scores)
    return weights @ V, weights

np.random.seed(42)
n, d = 6, 16
Q = np.random.randn(n, d)
K = np.random.randn(n, d)
V = np.random.randn(n, d)

mask = create_causal_mask(n)
out_causal, w_causal = masked_attention(Q, K, V, mask)

header('Exercise 3: Causal Masking')

# Check lower-triangular
upper = np.triu(w_causal, k=1)
check_true('attention matrix is lower-triangular', np.allclose(upper, 0, atol=1e-6))

# Check causality: modify position 5, position 0 should be unchanged
K_mod = K.copy()
K_mod[5] += 100
out_mod, _ = masked_attention(Q, K_mod, V, mask)
check_close('pos 0 unaffected by pos 5 change', out_causal[0], out_mod[0])

# But position 5 should change
check_true('pos 5 IS affected by its own change',
           not np.allclose(out_causal[5], out_mod[5]))

print('\nTakeaway: Causal masking enforces autoregressive structure — '
      'each position only sees its past.')

---

## Exercise 4: Rotary Position Embeddings (★★)

Implement RoPE and verify that attention scores depend only on relative position.

(a) Implement the 2D rotation matrix $R_m^{(i)}$
(b) Apply RoPE to query and key vectors
(c) Verify $(\mathbf{q}_m')^\top \mathbf{k}_n' = \mathbf{q}_m^\top R_{n-m} \mathbf{k}_n$
(d) Show score depends only on relative position $n - m$

In [ ]:
# Exercise 4: Your Solution

def apply_rope(x, pos, d_model, base=10000):
    """Apply RoPE rotation to vector x at position pos."""
    # YOUR CODE HERE
    x_rot = x.copy()
    # For each dimension pair (2i, 2i+1), apply rotation
    pass
    return x_rot

d = 16
q = np.random.randn(d)
k = np.random.randn(d)
result = apply_rope(q, 5, d)
print(result)

In [ ]:
# Exercise 4: Solution

def apply_rope(x, pos, d_model, base=10000):
    x_rot = x.copy()
    for i in range(d_model // 2):
        theta = pos / (base ** (2 * i / d_model))
        cos_t, sin_t = np.cos(theta), np.sin(theta)
        x0, x1 = x[2*i], x[2*i + 1]
        x_rot[2*i]     = x0 * cos_t - x1 * sin_t
        x_rot[2*i + 1] = x0 * sin_t + x1 * cos_t
    return x_rot

np.random.seed(42)
d = 16
q = np.random.randn(d)
k = np.random.randn(d)

header('Exercise 4: Rotary Position Embeddings')

# Test: same relative position gives same score
delta = 4
scores = []
for m in [0, 3, 7, 15, 42]:
    q_rot = apply_rope(q, m, d)
    k_rot = apply_rope(k, m + delta, d)
    scores.append(q_rot @ k_rot)

check_close(f'all scores equal for delta={delta}',
            np.array(scores), np.full(len(scores), scores[0]))

# Different delta gives different score
q_rot = apply_rope(q, 0, d)
k_rot2 = apply_rope(k, 2, d)
k_rot7 = apply_rope(k, 7, d)
check_true('different deltas give different scores',
           abs(q_rot @ k_rot2 - q_rot @ k_rot7) > 0.01)

# Verify rotation preserves norm
check_close('RoPE preserves ||q||', la.norm(apply_rope(q, 5, d)), la.norm(q))

print('\nTakeaway: RoPE encodes position via rotation — scores depend '
      'only on relative position, enabling length generalisation.')

---

## Exercise 5: Variance Scaling Verification (★★)

Verify the theorem: $\operatorname{Var}(\mathbf{q}^\top \mathbf{k}) = d_k$ for random unit-variance vectors.

(a) Generate 10,000 random query-key pairs for various $d_k$
(b) Compute empirical variance of unscaled and scaled dot products
(c) Plot results and compare with theory
(d) Show the effect on softmax entropy

In [ ]:
# Exercise 5: Your Solution

dims = [4, 16, 64, 128, 256, 512]
n_samples = 10000

for d_k in dims:
    q = np.random.randn(n_samples, d_k)
    k = np.random.randn(n_samples, d_k)
    dots = None  # YOUR CODE: compute q^T k for each sample
    var_unscaled = None  # YOUR CODE
    var_scaled = None    # YOUR CODE
    print(f'd_k={d_k}: var_unscaled={var_unscaled}, var_scaled={var_scaled}')

In [ ]:
# Exercise 5: Solution

dims = [4, 16, 64, 128, 256, 512]
n_samples = 10000
np.random.seed(42)

header('Exercise 5: Variance Scaling Verification')

vars_unscaled = []
vars_scaled = []
mean_entropies_unscaled = []
mean_entropies_scaled = []

for d_k in dims:
    q = np.random.randn(n_samples, d_k)
    k = np.random.randn(n_samples, d_k)
    dots = np.sum(q * k, axis=1)
    vars_unscaled.append(np.var(dots))
    vars_scaled.append(np.var(dots / np.sqrt(d_k)))

    # Entropy comparison (use first 100 samples as keys)
    K_sample = np.random.randn(20, d_k)
    ents_u, ents_s = [], []
    for i in range(min(200, n_samples)):
        scores = q[i] @ K_sample.T
        w_u = softmax(scores)
        w_s = softmax(scores / np.sqrt(d_k))
        ents_u.append(-np.sum(w_u * np.log(w_u + 1e-12)))
        ents_s.append(-np.sum(w_s * np.log(w_s + 1e-12)))
    mean_entropies_unscaled.append(np.mean(ents_u))
    mean_entropies_scaled.append(np.mean(ents_s))

for d_k, vu, vs in zip(dims, vars_unscaled, vars_scaled):
    print(f'd_k={d_k:4d}: Var(unscaled)={vu:8.2f} (theory={d_k}), '
          f'Var(scaled)={vs:.4f} (theory=1)')

check_true('Var(q^Tk) ~ d_k for all dims',
           all(abs(v - d) / d < 0.15 for v, d in zip(vars_unscaled, dims)))
check_true('Var(q^Tk/sqrt(d_k)) ~ 1 for all dims',
           all(abs(v - 1) < 0.2 for v in vars_scaled))

print(f'\nMean softmax entropy (d_k=512):')
print(f'  Unscaled: {mean_entropies_unscaled[-1]:.3f} (very low = saturated)')
print(f'  Scaled:   {mean_entropies_scaled[-1]:.3f} (healthy range)')
print(f'  Max possible: {np.log(20):.3f}')

print('\nTakeaway: Without 1/sqrt(d_k) scaling, softmax saturates as d_k grows — '
      'gradients vanish and attention cannot learn.')

---

## Exercise 6: Multi-Head Attention (★★)

Implement multi-head attention from single-head.

(a) Implement single-head attention as a function
(b) Split $d_{\text{model}} = 64$ into $h = 8$ heads with $d_k = 8$
(c) Create projection matrices and run all heads
(d) Concatenate and project with $W_O$
(e) Verify output shape

In [ ]:
# Exercise 6: Your Solution

def multi_head_attention(X, W_Qs, W_Ks, W_Vs, W_O, n_heads):
    """Multi-head attention.
    W_Qs: list of (d_model, d_k) projection matrices for queries
    """
    # YOUR CODE HERE
    pass

d_model = 64
n_heads = 8
d_k = d_model // n_heads
n_seq = 10
X = np.random.randn(n_seq, d_model)
print(f'Input shape: {X.shape}')

In [ ]:
# Exercise 6: Solution

def multi_head_attention(X, W_Qs, W_Ks, W_Vs, W_O, n_heads):
    heads = []
    for h in range(n_heads):
        Q_h = X @ W_Qs[h]
        K_h = X @ W_Ks[h]
        V_h = X @ W_Vs[h]
        d_k = Q_h.shape[-1]
        scores = Q_h @ K_h.T / np.sqrt(d_k)
        weights = softmax(scores)
        head_out = weights @ V_h
        heads.append(head_out)
    concat = np.concatenate(heads, axis=-1)
    return concat @ W_O

np.random.seed(42)
d_model = 64
n_heads = 8
d_k = d_model // n_heads
n_seq = 10

scale = 1.0 / np.sqrt(d_model)
W_Qs = [np.random.randn(d_model, d_k) * scale for _ in range(n_heads)]
W_Ks = [np.random.randn(d_model, d_k) * scale for _ in range(n_heads)]
W_Vs = [np.random.randn(d_model, d_k) * scale for _ in range(n_heads)]
W_O = np.random.randn(d_model, d_model) * scale

X = np.random.randn(n_seq, d_model)
output = multi_head_attention(X, W_Qs, W_Ks, W_Vs, W_O, n_heads)

header('Exercise 6: Multi-Head Attention')
print(f'Input shape:  {X.shape}')
print(f'Output shape: {output.shape}')
check_true('output shape matches input', output.shape == X.shape)

total_params = n_heads * 3 * d_model * d_k + d_model * d_model
expected = 4 * d_model**2
check_close('total params = 4d^2', total_params, expected)

print('\nTakeaway: Multi-head attention runs h independent attention ops '
      f'in parallel. Total params = 4d^2 = {expected:,}, independent of h={n_heads}.')

---

## Exercise 7: FlashAttention Online Softmax (★★★)

Implement the online softmax trick used in FlashAttention.

(a) Implement standard attention as reference
(b) Implement block-wise attention with online softmax
(c) Verify both give identical results
(d) Compare memory usage

In [ ]:
# Exercise 7: Your Solution

def flash_attention_simple(Q, K, V, block_size=2):
    """Block-wise attention with online softmax."""
    n, d_k = Q.shape
    d_v = V.shape[1]
    O = np.zeros((n, d_v))
    m = np.full(n, -np.inf)  # running max
    l = np.zeros(n)           # running sum-exp
    # YOUR CODE HERE: process K,V in blocks
    return O

n, d = 8, 16
Q = np.random.randn(n, d)
K = np.random.randn(n, d)
V = np.random.randn(n, d)
print('Implement block-wise attention here.')

In [ ]:
# Exercise 7: Solution

def standard_attention(Q, K, V):
    d_k = Q.shape[-1]
    S = Q @ K.T / np.sqrt(d_k)
    A = softmax(S)
    return A @ V

def flash_attention_simple(Q, K, V, block_size=2):
    n, d_k = Q.shape
    d_v = V.shape[1]
    O = np.zeros((n, d_v))
    m = np.full(n, -np.inf)
    l = np.zeros(n)

    for j0 in range(0, n, block_size):
        j1 = min(j0 + block_size, n)
        S_j = Q @ K[j0:j1].T / np.sqrt(d_k)
        m_new_block = np.max(S_j, axis=1)
        m_new = np.maximum(m, m_new_block)
        exp_old = np.exp(m - m_new)
        P_j = np.exp(S_j - m_new[:, None])
        l_new_block = np.sum(P_j, axis=1)
        l_new = exp_old * l + l_new_block
        O = (exp_old[:, None] * l[:, None] * O + P_j @ V[j0:j1]) / l_new[:, None]
        m = m_new
        l = l_new
    return O

np.random.seed(42)
n, d = 8, 16
Q = np.random.randn(n, d)
K = np.random.randn(n, d)
V = np.random.randn(n, d)

out_std = standard_attention(Q, K, V)
out_flash = flash_attention_simple(Q, K, V, block_size=2)

header('Exercise 7: FlashAttention Online Softmax')
diff = la.norm(out_std - out_flash)
check_true(f'Flash matches standard (diff={diff:.2e})', diff < 1e-10)

# Memory comparison
mem_standard = n * n  # full attention matrix
mem_flash = n * 2  # only block_size columns at a time + m, l
print(f'\nStandard memory: O(n^2) = {mem_standard} elements')
print(f'Flash memory:    O(n) = ~{mem_flash} elements')
check_true('Flash uses less memory', mem_flash < mem_standard)

print('\nTakeaway: FlashAttention computes exact attention in O(n) memory '
      'using the online softmax trick — no approximation needed.')

---

## Exercise 8: LoRA Adaptation (★★★)

(a) Create a pretrained weight $W_0 \in \mathbb{R}^{64 \times 64}$
(b) Add LoRA with rank $r = 4$: $\Delta W = BA$
(c) Verify the output changes from the base model
(d) Compute parameter reduction ratio
(e) Verify $\Delta W$ has rank exactly $r$

In [ ]:
# Exercise 8: Your Solution

d = 64
r = 4
W0 = np.random.randn(d, d) * 0.02

# YOUR CODE: create A (r, d) and B (d, r)
A = None
B = None
delta_W = None  # B @ A

print('Implement LoRA here.')

In [ ]:
# Exercise 8: Solution

np.random.seed(42)
d = 64
r = 4
W0 = np.random.randn(d, d) * 0.02

# LoRA matrices
A = np.random.randn(r, d) / np.sqrt(d)
B = np.zeros((d, r))  # init to zero

header('Exercise 8: LoRA Adaptation')

# At init: delta_W = 0
delta_W_init = B @ A
check_true('At init, delta_W = 0', np.allclose(delta_W_init, 0))

# Simulate training: update B
B = np.random.randn(d, r) * 0.01
delta_W = B @ A
W_adapted = W0 + delta_W

# Verify rank
U, S, Vt = la.svd(delta_W)
rank = np.sum(S > 1e-10)
check_true(f'rank(delta_W) = {rank} (expected {r})', rank == r)

# Forward pass comparison
x = np.random.randn(d)
out_base = x @ W0.T
out_adapted = x @ W_adapted.T
out_lora = x @ W0.T + x @ A.T @ B.T  # equivalent decomposed form

check_close('W_adapted forward == decomposed LoRA forward',
            out_adapted, out_lora)
check_true('output changed from base', not np.allclose(out_base, out_adapted))

# Parameter savings
full_params = d * d
lora_params = 2 * r * d
print(f'\nFull fine-tuning: {full_params:,} params')
print(f'LoRA (r={r}):      {lora_params:,} params')
print(f'Reduction:         {full_params // lora_params}x')

# SVD spectrum
print(f'\nSingular values of delta_W: {S[:6]}')
print(f'Rank-{r}: only {r} non-zero singular values')

print('\nTakeaway: LoRA adds a rank-r update to frozen weights — '
      f'{full_params//lora_params}x fewer parameters while capturing the '
      'low-rank structure of weight updates during fine-tuning.')

---

## What to Review After Finishing

- [ ] Can you derive the $1/\sqrt{d_k}$ scaling from the variance of dot products?
- [ ] Can you explain why multi-head attention has $4d^2$ parameters regardless of head count?
- [ ] Do you understand how RoPE makes attention scores depend only on relative position?
- [ ] Can you implement the online softmax trick and explain why it saves memory?
- [ ] Do you understand how LoRA exploits the low-rank structure of weight updates?

## References

1. Vaswani et al. (2017) "Attention Is All You Need"
2. Su et al. (2021) "RoFormer: Enhanced Transformer with Rotary Position Embedding"
3. Dao et al. (2022) "FlashAttention: Fast and Memory-Efficient Exact Attention"
4. Hu et al. (2022) "LoRA: Low-Rank Adaptation of Large Language Models"
5. Touvron et al. (2023) "LLaMA: Open and Efficient Foundation Language Models"